# Project 3: Exploration in reinforcement learning

This notebook is the starting point of the project; the subject (`projects.pdf`, section 3)
gives the context and the grading. You hand in this notebook, completed: it must run from
top to bottom in less than 15 minutes on a laptop.

The project has three guided tasks, then a part where you choose how to go further:

1. **$\varepsilon$-greedy fails**: run Q-learning on two MDPs, and explain the failure.
2. **Optimism in Q-learning**: optimistic initialization and a count bonus.
3. **Optimism in a model**: learn the transitions, and plan on the learned model with a
   bonus.
4. **Going further**: a direction of your choice (see the end of the notebook).

Each task ends with a figure and a short written answer (a few sentences) in the cell
"Answer". Everything outside the cells marked `...  # TODO` is given and working.

This notebook needs the version of `seqlearn` that contains `projects/`: if the imports below
fail, download the setup archive again from the course page and replace the folder
`seqlearn/`.

In [ ]:
# Checks that your folder seqlearn/ is recent enough, and updates it otherwise. (Provided, just run it.)
import importlib.util, io, pathlib, re, shutil, sys, urllib.request, zipfile

REQUIRED = "2026.10.30"
URL = "https://igorcolin.github.io/teaching/sequential-decision/sequential-decision-setup.zip"


def seqlearn_version():
    spec = importlib.util.find_spec("seqlearn")
    if spec is None or spec.origin is None:
        return None, pathlib.Path("seqlearn")
    init = pathlib.Path(spec.origin)
    match = re.search(r'^__version__ = "([^"]+)"', init.read_text(), re.MULTILINE)
    return (match.group(1) if match else "none"), init.parent


version, folder = seqlearn_version()
if version is not None and version != "none" and version >= REQUIRED:
    print(f"seqlearn {version}: up to date")
else:
    print(f"seqlearn {version or 'not found'}: version {REQUIRED} or later needed, downloading it")
    try:
        data = urllib.request.urlopen(URL, timeout=30).read()
    except OSError as error:
        raise RuntimeError(f"download failed ({error}): download {URL} by hand, unzip it, "
                           "and replace your folder seqlearn/ with the new one") from None
    old = folder.with_name("seqlearn.old")
    if folder.exists():
        shutil.rmtree(old, ignore_errors=True)
        folder.rename(old)
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        for name in archive.namelist():
            parts = pathlib.PurePosixPath(name).parts
            if len(parts) > 2 and parts[1] == "seqlearn" and not name.endswith("/"):
                target = folder.joinpath(*parts[2:])
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(name))
    importlib.invalidate_caches()
    version = seqlearn_version()[0]
    if version == "none" or version < REQUIRED:
        raise RuntimeError(f"the course page still has seqlearn {version}: tell your teacher")
    print(f"seqlearn updated to {version} (previous folder kept as {old})")
    if "seqlearn" in sys.modules:
        print("Restart the kernel (Kernel > Restart), then run the notebook from the top.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from seqlearn.plotting import COLORS, plot_mean, set_style
from seqlearn.projects.explore import EPISODE_LENGTH, GAMMA, make_riverswim, make_slippery_maze, policy_value

set_style()

## 0. The two MDPs

Both MDPs have the interface of Lab 7: `env.reset()`, `env.step(a)` returns
`(next state, reward, done)`, and the arrays `env.P`, `env.R` (for the evaluation only: the
agents must not read them). Episodes last at most $50$ steps, and $\gamma = 0.95$.

- **RiverSwim** ($6$ states in a row, actions left and right). Left always moves downstream,
  and gives a small reward $0.005$ in the first state. Right swims upstream: it moves right
  with probability $0.35$ only, and gives a reward $1$ in the last state.
- **Slippery maze**: a small reward $0.05$ (g) next to the start, the goal G ($+1$) far away;
  each move slips in a random direction with probability $0.2$.

In [ ]:
river, maze = make_riverswim(), make_slippery_maze()
print(f"RiverSwim: {river.n_states} states, {river.n_actions} actions")
fig, ax = plt.subplots(figsize=(6, 3))
maze.plot(ax, title=f"Slippery maze: {maze.n_states} states")
plt.show()

## Given: value iteration, training and evaluation

`value_iteration(P, R)` returns $V$ and $Q$ (Lab 7). An agent has two methods: `act(s)`
returns an action, and `update(s, a, r, s_next, done)` learns from a transition; its
attribute `policy()` returns the greedy policy it would deploy. `learning_curve` trains an
agent for $1000$ episodes and measures, every $50$ episodes, the value at the start state of
its greedy policy, divided by the optimal value: $1$ means optimal.

In [ ]:
def value_iteration(P, R, gamma=GAMMA, n_iter=300, V=None):
    V = np.zeros(P.shape[0]) if V is None else V
    for _ in range(n_iter):
        Q = R + gamma * P @ V
        V = Q.max(axis=1)
    return V, Q


class QLearning:
    def __init__(self, n_states, n_actions, epsilon=0.1, alpha=0.1, init=0.0, rng=None):
        self.epsilon, self.alpha = epsilon, alpha
        self.Q = np.full((n_states, n_actions), init)
        self.rng = np.random.default_rng(rng)

    def greedy(self, values):
        return int(self.rng.choice(np.flatnonzero(values == values.max())))

    def act(self, s):
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.Q.shape[1]))
        return self.greedy(self.Q[s])

    def update(self, s, a, r, s_next, done):
        target = r if done else r + GAMMA * self.Q[s_next].max()
        self.Q[s, a] += self.alpha * (target - self.Q[s, a])

    def policy(self):
        return self.Q.argmax(axis=1)


def learning_curve(make_env, make_agent, seed, n_episodes=1000, every=50):
    env = make_env(rng=seed)
    agent = make_agent(env, np.random.default_rng(seed))
    v_star = value_iteration(env.P, env.R, n_iter=2000)[0][env.start]
    curve = []
    for episode in range(n_episodes):
        s = env.reset()
        for _ in range(EPISODE_LENGTH):
            a = agent.act(s)
            s_next, r, done = env.step(a)
            agent.update(s, a, r, s_next, done)
            s = s_next
            if done:
                break
        if (episode + 1) % every == 0:
            curve.append(policy_value(env, agent.policy())[env.start] / v_star)
    return np.array(curve)


def plot_curves(curves, title, n_episodes=1000, every=50):
    """curves: dictionary (MDP name) -> dictionary (agent name) -> array (n_seeds, n_points)."""
    fig, axes = plt.subplots(1, len(curves), figsize=(5.5 * len(curves), 3.2), squeeze=False)
    for ax, (mdp, by_agent) in zip(axes[0], curves.items()):
        x = np.arange(every, n_episodes + 1, every)
        for (name, c), color in zip(by_agent.items(), COLORS):
            for run in c:
                ax.plot(x, run, color=color, alpha=0.2, linewidth=1)
            ax.plot(x, c.mean(axis=0), color=color, label=name)
        ax.set(xlabel="episode", ylabel="value of the greedy policy / optimal", ylim=(-0.05, 1.05), title=f"{title}: {mdp}")
        ax.legend(loc="center right")
    plt.show()


MDPS = {"RiverSwim": make_riverswim, "slippery maze": make_slippery_maze}
SEEDS = range(5)

## Task 1: $\varepsilon$-greedy fails

Run Q-learning with $\varepsilon = 0.1$, $\alpha = 0.1$ and $Q = 0$ at the start, on both MDPs
and $5$ seeds, and plot the learning curves.

**Expected**: the figure, and an answer to: which policy does Q-learning learn on each MDP?
Explain why, with the structure of the MDP. For RiverSwim, estimate the probability that a
random walk with $\varepsilon$-greedy reaches the last state in an episode. (Hint: the
reward $0.005$ is found at the first step, and from then on the greedy action is left.)

**To read**: slides of session 7 (Q-learning, exploration); Sutton and Barto, *Reinforcement
Learning: An Introduction*, 2nd edition, 2018, sections 2.6 (optimistic initial values) and
6.5 (Q-learning) (free online).

In [ ]:
curves1 = {}
...  # TODO
plot_curves(curves1, "Task 1")

**Answer.** *Write your answer here.*

## Task 2: optimism in Q-learning

Two ways of making Q-learning optimistic:

- **optimistic initialization**: $Q = 1/(1 - \gamma) = 20$ at the start, an upper bound on
  any value since the rewards are at most $1$;
- **a count bonus**: learn with the reward $r + c / \sqrt{N(s, a)}$, where $N(s, a)$ counts
  the visits of $(s, a)$, so that rarely tried actions look good.

Complete `BonusQLearning`, then compare on both MDPs and $5$ seeds: $\varepsilon$-greedy
(Task 1), optimistic initialization ($\varepsilon = 0$), and the bonus with $c = 1$
($\varepsilon = 0.1$).

**Expected**: the figure, and an answer to: which method solves RiverSwim? Is the maze
solved within $1000$ episodes? Why is it harder for Q-learning? (Hint: how many updates does
the information "G is good" need to travel from G to the start?)

**To read**: Strehl and Littman, *An analysis of model-based interval estimation for Markov
decision processes*, JCSS 2008, section 1 and the definition of the bonus.

In [ ]:
class BonusQLearning(QLearning):
    def __init__(self, n_states, n_actions, c=1.0, **kwargs):
        super().__init__(n_states, n_actions, **kwargs)
        self.c = c
        self.N = np.zeros((n_states, n_actions))

    def update(self, s, a, r, s_next, done):
        ...  # TODO

In [ ]:
agents2 = {"epsilon-greedy": lambda env, rng: QLearning(env.n_states, env.n_actions, rng=rng)}
...  # TODO
curves2 = {mdp: {name: np.array([learning_curve(make_env, make_agent, seed) for seed in SEEDS])
                 for name, make_agent in agents2.items()} for mdp, make_env in MDPS.items()}
plot_curves(curves2, "Task 2")

**Answer.** *Write your answer here.*

## Task 3: optimism in a model

A model-based agent (MBIE-EB, Strehl and Littman 2008) counts its transitions, and before
each episode:

1. estimates $\hat P(s' \mid s, a) = N(s, a, s') / N(s, a)$ and
   $\hat r(s, a) = (\text{sum of the rewards of } (s, a)) / N(s, a)$;
2. adds the bonus $\beta / \sqrt{N(s, a)}$ to $\hat r(s, a)$;
3. for the pairs never tried ($N(s, a) = 0$), uses a self-loop with reward $1$: the most
   optimistic assumption; except in the terminal states met so far (`self.terminal`), where
   nothing happens anymore (no transition, reward $0$);
4. runs value iteration on this model, and plays greedily during the episode.

Complete `plan` in `MBIE`, then run it with $\beta = 0.2$ on both MDPs and $5$ seeds; then on
the maze only, with $\beta \in \{0.02, 0.2, 2\}$.

**Expected**: the two figures, and an answer to: why does the model-based agent learn so much
faster than Q-learning on the maze? What does $\beta$ change, and what happens if it is too
small or too large? What is the cost in computation?

**To read**: Strehl and Littman, JCSS 2008 (MBIE-EB, section 3); Osband, Russo and Van Roy,
*(More) efficient reinforcement learning via posterior sampling*, NeurIPS 2013 (the
Thompson sampling of MDPs, to go further).

In [ ]:
class MBIE:
    def __init__(self, n_states, n_actions, beta=0.2, rng=None):
        self.beta = beta
        self.counts = np.zeros((n_states, n_actions, n_states))
        self.reward_sums = np.zeros((n_states, n_actions))
        self.terminal = np.zeros(n_states, dtype=bool)
        self.rng = np.random.default_rng(rng)
        self.V = None
        self.plan()

    def model(self, bonus):
        """Estimated (P, R), with the bonus beta / sqrt(N) if bonus is True."""
        S, A = self.reward_sums.shape
        N = self.counts.sum(axis=2)
        ...  # TODO
        return P, R

    def plan(self):
        P, R = self.model(bonus=True)
        self.V, self.Q = value_iteration(P, R, n_iter=200, V=self.V)

    def act(self, s):
        return int(self.rng.choice(np.flatnonzero(self.Q[s] == self.Q[s].max())))

    def update(self, s, a, r, s_next, done):
        self.counts[s, a, s_next] += 1
        self.reward_sums[s, a] += r
        self.terminal[s_next] |= done
        if done or self.counts.sum() % EPISODE_LENGTH == 0:
            self.plan()

    def policy(self):
        P, R = self.model(bonus=False)
        return value_iteration(P, R)[1].argmax(axis=1)

In [ ]:
...  # TODO
plot_curves(curves3, "Task 3")

In [ ]:
...  # TODO
plot_curves(curves3b, "Task 3")

**Answer.** *Write your answer here.*

## Going further

Choose one direction, among these suggestions or your own idea: state the question you ask,
then answer it as in the tasks (code, a figure, a short answer). One well-studied direction
is better than several superficial ones.

- Posterior sampling (PSRL, Osband et al. 2013): sample a model from a Dirichlet posterior
  before each episode instead of adding a bonus.
- Schedules of $\varepsilon$ (decreasing with the episodes): can they solve RiverSwim?
- A longer river (`make_riverswim(n_states=12)`): how does the number of episodes needed
  grow with the length, for each method?